# Seedes futások, LFCC – Janus 0.2 és 0.3, seed 0/1/2

**Egyszer kell beállítani, utána csak Save Version → Save & Run All.**

Jobb oldali panel:
1. **Accelerator** → GPU T4 x2
2. **Internet** → On
3. **Input → Add Input** → `asvpoof-2019-dataset` (awsaf49)

6 futás: Janus 0.2 (janus_01, `config.yaml`, LFCC) és Janus 0.3 (janus_03), seed 0, 1, 2. Csak a seed új; config, 30 epoch, batch 64, lr 1e-3, cosine, augmentáció ugyanaz, mint Run 4 és Run 5.

A két repó feature-kódja és configja azonos, ezért az LFCC-cache (`/tmp/cache`) közös: csak az első futás építi fel (train/dev ~14 perc, eval ~15 perc). Várható idő összesen kb. 4–5 óra (a Kaggle-munkamenet felső határa 12 óra).

A sorrend seed szerint halad (0.2 seed0, 0.3 seed0, 0.2 seed1, ...), hogy ha valami megszakad, a meglévő futások kiegyensúlyozottak legyenek.

Kimenet futásonként: `/kaggle/working/out/v02_seed0` … `v03_seed2`, mindegyikben `train_log.txt`, `janus_best.pt`, `eval_scores.csv`, `eval_per_attack.csv`, `commit.txt`. A végén `out/summary.md` (átlag ± szórás). A teljes `out` mappát töltsd le az **Output** fülről.

## 0. Környezet ellenőrzése

In [ ]:
import os, sys, torch
from pathlib import Path

print('torch', torch.__version__)
assert torch.cuda.is_available(), 'NINCS GPU. Jobb oldali panel → Accelerator → GPU T4 x2, majd Run All újra.'
print('GPU:', torch.cuda.get_device_name(0))

inputs = list(Path('/kaggle/input').iterdir())
print('/kaggle/input:', [p.name for p in inputs])
assert inputs, 'NINCS ADAT. Jobb oldali panel → Input → Add Input → "asvpoof-2019-dataset" (awsaf49), majd Run All újra.'

## 1. Kód behúzása GitHubról (janus_01 és janus_03)

In [ ]:
REPOS = [('janus_01', '/kaggle/working/janus_01'), ('janus_03', '/kaggle/working/janus_03')]
SEEDS = [0, 1, 2]
# (kimeneti név, repó, repó útvonal, config, elvárt feature)
RUNS = [
    ('v02', 'janus_01', '/kaggle/working/janus_01', 'config.yaml', 'lfcc'),
    ('v03', 'janus_03', '/kaggle/working/janus_03', 'config.yaml', 'lfcc'),
]

In [ ]:
COMMIT = {}
for name, path in REPOS:
    if not os.path.exists(path):
        !git clone -q https://github.com/Csaszi05/{name}.git {path}
    else:
        !cd {path} && git pull -q
    head = !git -C {path} rev-parse --short HEAD
    COMMIT[name] = head[0]
    !git -C {path} log --oneline | head -3
print(COMMIT)
!pip install -q -r {REPOS[0][1]}/requirements.txt

## 2. Adat megkeresése és ellenőrzése

Az adathalmaz gyökere az a mappa, amelyben az `ASVspoof2019_LA_cm_protocols` könyvtár van. A cella megkeresi a `/kaggle/input` alatt, bárhogy is van becsomagolva.

In [ ]:
def find_dir(root, name, depth=8):
    # szélességi keresés, max 8 szint mélyen; nem járja be a flac fájlokat
    level = [Path(root)]
    for _ in range(depth):
        nxt = []
        for d in level:
            for p in d.iterdir():
                if not p.is_dir():
                    continue
                if p.name == name:
                    return p
                nxt.append(p)
        level = nxt
    return None

proto = find_dir('/kaggle/input', 'ASVspoof2019_LA_cm_protocols')
assert proto, 'Nem találom az ASVspoof2019_LA_cm_protocols mappát a /kaggle/input alatt.'
DATA_ROOT = str(proto.parent); print('DATA_ROOT  =', DATA_ROOT)
!ls {DATA_ROOT}

sys.path.insert(0, f'{REPOS[0][1]}/src')
from dataset import load_protocol
tr = load_protocol(DATA_ROOT, 'train'); dv = load_protocol(DATA_ROOT, 'dev')
print('train', len(tr), 'spoof arány', round(tr.y.mean(), 3))
print('dev  ', len(dv), 'spoof arány', round(dv.y.mean(), 3))
assert os.path.exists(tr.path[0]), f'A protokoll megvan, de a flac fájl nem: {tr.path[0]}'

## 3. Tanítás és kiértékelés, seedenként

Minden futás után a checkpointból ellenőrzi, hogy a seed és a feature az elvárt. Ha egy futás elhasal, a többi megy tovább; a cella végén kiírja a hibás futásokat.

In [ ]:
OUT = Path('/kaggle/working/out')
failed = []
for seed in SEEDS:
    for tag, repo, path, config, feature in RUNS:
        run = OUT / f'{tag}_seed{seed}'
        run.mkdir(parents=True, exist_ok=True)
        (run / 'commit.txt').write_text(f'{repo} {COMMIT[repo]} {config} seed={seed}\n')
        print(f'\n===== {run.name}  ({repo} {COMMIT[repo]}, {config}) =====', flush=True)
        train_cmd = (f'cd {path} && python src/train.py --data_root {DATA_ROOT} --config {config} '
                     f'--epochs 30 --augment --sched cosine --batch 64 --lr 1e-3 --workers 2 '
                     f'--seed {seed} --cache_dir /tmp/cache --out {run}')
        print(train_cmd, flush=True)
        !{train_cmd}
        if _exit_code != 0:
            failed.append((run.name, 'train')); continue
        ck = torch.load(run / 'janus_best.pt', map_location='cpu', weights_only=False)
        assert ck['seed'] == seed and ck['config']['audio']['feature'] == feature, (ck['seed'], ck['config']['audio'])
        print('legjobb epoch:', ck['epoch'], flush=True)
        eval_cmd = (f'cd {path} && python src/evaluate.py --data_root {DATA_ROOT} '
                    f'--checkpoint {run}/janus_best.pt --split eval --workers 2 --cache_dir /tmp/cache --out {run}')
        !{eval_cmd}
        if _exit_code != 0:
            failed.append((run.name, 'eval'))
print('\nhibás futások:', failed or 'nincs')

## 4. Gyors összefoglaló és summary.md

In [ ]:
import pandas as pd
rows = []
for run in sorted(OUT.glob('*_seed*')):
    if (run / 'eval_per_attack.csv').exists():
        log = (run / 'train_log.txt').read_text().splitlines()
        rows.append({'run': run.name, 'best_dev': log[-1].split()[-1],
                     'attack_mean_eer_%': round(pd.read_csv(run / 'eval_per_attack.csv').eer.mean() * 100, 2)})
print(pd.DataFrame(rows).to_string(index=False))
!ls -la {OUT}/*

In [ ]:
!python /kaggle/working/janus_03/scripts/summarize_seeds.py --runs {OUT} --out {OUT}
!cat {OUT}/summary.md